In [1]:
import os
from scipy.io import loadmat
import numpy as np
import pandas as pd

In [2]:
mat_dir = "../datasets/RecoFit"

files = [f for f in os.listdir(mat_dir) if f.endswith(".mat")]
print(f"Found {len(files)} mat files.")


Found 126 mat files.


Data format

accel: time, x, y, z
gyro: time, x, y, z
act_start: time
act_end: time
act_name: string
act_start.shape = act_end.shape = act_name.shape

In [3]:
datamat = loadmat(os.path.join(mat_dir, files[0]))
datamat

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Mon Dec  1 22:45:45 2025',
 '__version__': '1.0',
 '__globals__': [],
 'accel': array([[ 0.00000000e+00, -1.05339641e+00, -2.40851366e-01,
          7.64406976e-03],
        [ 1.99999531e-02, -1.03511169e+00, -2.06829100e-01,
         -1.54041625e-02],
        [ 3.99999062e-02, -9.71612032e-01, -1.75238200e-01,
          8.48188294e-03],
        ...,
        [ 2.55911400e+03,  3.45431196e-02, -6.49887318e-02,
          8.78636179e-01],
        [ 2.55913400e+03,  3.18584003e-02, -7.04902794e-02,
          8.99530284e-01],
        [ 2.55915400e+03,  2.24590146e-02, -4.43846482e-02,
          6.28852408e-01]], shape=(127959, 4)),
 'act_end': array([[ 103.019],
        [ 107.632],
        [ 108.859],
        [ 131.491],
        [ 154.023],
        [ 187.307],
        [ 229.758],
        [ 369.624],
        [ 399.66 ],
        [ 475.593],
        [ 561.128],
        [ 716.538],
        [ 769.146],
        [ 805.023],
     

In [8]:
dfs = []

for i, f in enumerate(files):
    datamat = loadmat(os.path.join(mat_dir, f))

    recordingID = datamat["recordingID"]
    recordingID = recordingID[0][0]

    subjectID = datamat["subjectID"]
    subjectID = subjectID[0][0]

    accel = datamat["accel"]
    gyro = datamat["gyro"]

    act_start = datamat["act_start"]
    act_start = act_start.reshape(-1)

    act_end = datamat["act_end"]
    act_end = act_end.reshape(-1)

    act_name = datamat["act_name"]
    act_name = act_name.reshape(-1)
    act_name = np.array([x[0] for x in act_name])

    time = accel[:, 0]
    activity = np.full(time.shape, None, dtype=object)
    for s, e, name in zip(act_start, act_end, act_name):
        mask = (time >= s) & (time <= e) # creates a true/false mask for time
        activity[mask] = name # wherever mask is true set activity to this name

    activity = pd.Series(activity).fillna("Unknown").to_numpy()

    dfi = pd.DataFrame({
        "acc_x": accel[:, 1],
        "acc_y": accel[:, 2],
        "acc_z": accel[:, 3],
        "gyr_x": gyro[:, 1],
        "gyr_y": gyro[:, 2],
        "gyr_z": gyro[:, 3],
        "activity": activity,
        "trainer": i,
        # "subjectID": subjectID,
        # "recordingID": recordingID,
        "time": time,
    })

    dfs.append(dfi)

print(i)
data_df = pd.concat(dfs)


125


In [9]:
# 1) build activity -> int index (sorted for stability)
activity_index = {str(act): i for i, act in enumerate(sorted(data_df["activity"].unique()))}

# 2) replace activity labels with ints
data_df["activity"] = data_df["activity"].map(activity_index).astype("int32")

activity_index


{'<Initial Activity>': 0,
 'Alternating Dumbbell Curl': 1,
 'Arm Band Adjustment': 2,
 'Arm straight up': 3,
 'Band Pull-Down Row': 4,
 'Bicep Curl': 5,
 'Biceps Curl (band)': 6,
 'Box Jump (on bench)': 7,
 'Burpee': 8,
 'Butterfly Sit-up': 9,
 'Chest Press (rack)': 10,
 'Crunch': 11,
 'Device on Table': 12,
 'Dip': 13,
 'Dumbbell Deadlift Row': 14,
 'Dumbbell Row (knee on bench) (label spans both arms)': 15,
 'Dumbbell Row (knee on bench) (left arm)': 16,
 'Dumbbell Row (knee on bench) (right arm)': 17,
 'Dumbbell Squat (hands at side)': 18,
 'Dynamic Stretch (at your own pace)': 19,
 'Elliptical machine': 20,
 'Fast Alternating Punches': 21,
 'Invalid': 22,
 'Jump Rope': 23,
 'Jumping Jacks': 24,
 'Kettlebell Swing': 25,
 'Lateral Raise': 26,
 'Lawnmower (label spans both arms)': 27,
 'Lawnmower (left arm)': 28,
 'Lawnmower (right arm)': 29,
 'Lunge (alternating both legs, weight optional)': 30,
 'Medicine Ball Slam': 31,
 'Non-Exercise': 32,
 'Note': 33,
 'Overhead Triceps Extension

In [13]:
ACTIVITY_MAPPING = {activity_index[i]: i for i in activity_index}
ACTIVITY_MAPPING

{0: '<Initial Activity>',
 1: 'Alternating Dumbbell Curl',
 2: 'Arm Band Adjustment',
 3: 'Arm straight up',
 4: 'Band Pull-Down Row',
 5: 'Bicep Curl',
 6: 'Biceps Curl (band)',
 7: 'Box Jump (on bench)',
 8: 'Burpee',
 9: 'Butterfly Sit-up',
 10: 'Chest Press (rack)',
 11: 'Crunch',
 12: 'Device on Table',
 13: 'Dip',
 14: 'Dumbbell Deadlift Row',
 15: 'Dumbbell Row (knee on bench) (label spans both arms)',
 16: 'Dumbbell Row (knee on bench) (left arm)',
 17: 'Dumbbell Row (knee on bench) (right arm)',
 18: 'Dumbbell Squat (hands at side)',
 19: 'Dynamic Stretch (at your own pace)',
 20: 'Elliptical machine',
 21: 'Fast Alternating Punches',
 22: 'Invalid',
 23: 'Jump Rope',
 24: 'Jumping Jacks',
 25: 'Kettlebell Swing',
 26: 'Lateral Raise',
 27: 'Lawnmower (label spans both arms)',
 28: 'Lawnmower (left arm)',
 29: 'Lawnmower (right arm)',
 30: 'Lunge (alternating both legs, weight optional)',
 31: 'Medicine Ball Slam',
 32: 'Non-Exercise',
 33: 'Note',
 34: 'Overhead Triceps Exten

In [7]:
data_df

,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,activity,trainer,time
0,-1.053396,-0.240851,0.007644,-6.320330,-38.337973,-16.067351,0,0,0.000
1,-1.035112,-0.206829,-0.015404,2.997305,-49.883744,-17.257456,0,0,0.020
2,-0.971612,-0.175238,0.008482,2.541084,-61.131924,-16.663252,0,0,0.040
3,-0.905299,-0.147946,0.044365,-8.750498,-67.743795,-14.327861,0,0,0.060
4,-0.887257,-0.135992,0.091578,-21.993410,-69.473844,-11.848797,0,0,0.080
...,...,...,...,...,...,...,...,...,...
127914,-0.261000,0.351041,0.807325,-6.851228,3.066831,-3.437496,12,125,2558.269
127915,-0.263873,0.352947,0.809510,-6.891108,3.072903,-3.451624,12,125,2558.289
127916,-0.260786,0.350253,0.804076,-6.906065,3.132749,-3.482305,12,125,2558.309
127917,-0.267537,0.358723,0.836679,-7.173825,3.200092,-3.632732,12,125,2558.329
